In [1]:

!nvidia-smi
!nvcc --version

Sun Oct 11 14:59:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_cuda.cu
#include <iostream>
#include <cstdlib>
#include <cuda_runtime.h>
#include <random>
#include <vector>
#include <cmath>
#include <algorithm>

using namespace std;

// check whether the cuda operation succeeded.
// we wrap all cuda operations in this to handle failures at any point
void checkCuda(cudaError_t result) {
    if (result != cudaSuccess) {
        cerr << cudaGetErrorString(result) << endl;
        exit(EXIT_FAILURE);
    }
}

__global__ void matmul(
    const float* A,
    const float* B,
    float* C,
    int M, int K, int N
) {
    // giving the thread its dimensions to execute
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    int j = blockIdx.x * blockDim.x + threadIdx.x;


    if (i < M && j < N) {
        float sum = 0;

        // computing dot product
        for (int k = 0; k < K; k++) {
            sum += A[i*K + k] * B[k*N + j];
        }

        C[i*N + j] = sum;
    }
}

// basic matrix multiplication, used to compare to gpu matmul
void cpuMatmul(
    const float* A,
    const float* B,
    float* C,
    int M, int K, int N
) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;

            for (int k = 0; k < K; k++) {
                sum += A[i*K + k] * B[k*N + j];
            }

            C[i*N + j] = sum;
        }
    }
}

int main() {
    // Using example matrices; they can be arbitrarily large
    const int M = 55;
    const int K = 66;
    const int N = 77;

    unsigned int seed = 100;
    mt19937 gen(seed);
    uniform_real_distribution<float> distrib(-1.0f, 1.0f);

    vector<float> h_A(M*K);
    vector<float> h_B(K*N);
    vector<float> h_C(M*N);

    for (int i = 0; i < M; i++){
      for (int j = 0; j < K; j++){
        h_A[i * K + j] = distrib(gen);
      }
    }

    for (int i = 0; i < K; i++){
      for (int j = 0; j < N; j++){
        h_B[i * N + j] = distrib(gen);
      }
    }


    float* d_A = nullptr;
    float* d_B = nullptr;
    float* d_C = nullptr;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    checkCuda(cudaMalloc((void**)&d_A, bytesA));
    checkCuda(cudaMalloc((void**)&d_B, bytesB));
    checkCuda(cudaMalloc((void**)&d_C, bytesC));

    checkCuda(cudaMemcpy(d_A, h_A.data(), bytesA, cudaMemcpyHostToDevice));
    checkCuda(cudaMemcpy(d_B, h_B.data(), bytesB, cudaMemcpyHostToDevice));

    // initializing block of threads
    // most efficient size depends on the size of input matrix
    dim3 block(16, 16);

    dim3 grid(
        (N + block.x - 1) / block.x,
        (M + block.y - 1) / block.y
    );


    // executes on the GPU
    matmul<<<grid, block>>>(d_A, d_B, d_C, M, K, N);

    checkCuda(cudaGetLastError());
    checkCuda(cudaDeviceSynchronize());

    // copy result from GPU back to CPU
    checkCuda(cudaMemcpy(h_C.data(), d_C, bytesC, cudaMemcpyDeviceToHost));

    // create and compute an array to store the CPU reference result
    std::vector<float> expected(M * N);
    cpuMatmul(h_A.data(), h_B.data(), expected.data(), M, K, N);

    // now, calculate the error in our GPU-calculated matrix
    const float atol = 1e-3f;
    const float rtol = 1e-4f;

    float maxError = 0.0f;
    int failures = 0;

    for (int i = 0; i < M * N; i++) {
        float gpuValue = h_C[i];
        float cpuValue = expected[i];

        float error = fabs(gpuValue - cpuValue);

        float tolerance = atol + rtol * fabs(cpuValue);

        // track the largest absolute error
        if (std::isfinite(error)) {
            maxError = max(maxError, error);
        }

        // check for invalid numbers or excessive error
        if (!std::isfinite(gpuValue) ||
            !std::isfinite(cpuValue) ||
            error > tolerance) {

            failures++;

            // Print only the first few failures.
            if (failures <= 5) {
                std::cout << "Mismatch at ("
                          << i / N << ", " << i % N << "): "
                          << "CPU = " << cpuValue
                          << ", GPU = " << gpuValue
                          << std::endl;
            }
        }
    }

    std::cout << "Maximum absolute error: "
              << maxError << std::endl;

    if (failures == 0) {
        std::cout << "PASS: All values match within tolerance."
                  << std::endl;
    } else {
        std::cout << "FAIL: " << failures
                  << " values did not match."
                  << std::endl;
    }

    checkCuda(cudaFree(d_A));
    checkCuda(cudaFree(d_B));
    checkCuda(cudaFree(d_C));

    return failures == 0 ? 0: 1;
}


Writing matmul_cuda.cu


In [5]:
!nvcc matmul_cuda.cu -o matmul_cuda
!./matmul_cuda

Maximum absolute error: 2.38419e-06
PASS: All values match within tolerance.


In [ ]:

import os
import subprocess
from google.colab import userdata

repo = "/content/cuda-matmul-optimization"

# Retrieve your existing token from Colab Secrets.
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing.")

# Set up authentication for these commands.
env = os.environ.copy()
env["GH_TOKEN"] = token

# Configure Git to use GitHub CLI.
subprocess.run(
    ["gh", "auth", "setup-git",
     "--hostname", "github.com", "--force"],
    cwd=repo,
    env=env,
    check=True
)

# Push your already-created commit.
result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("SUCCESS! Your latest commit is on GitHub.")
else:
    print("Push failed:")
    print(result.stderr.replace(token, "[REDACTED]"))


In [8]:
%cd /content
!git clone https://github.com/loganKneller/cuda-matmul-optimization.git
%cd /content/cuda-matmul-optimization
!git log --oneline -5


/content
fatal: destination path 'cuda-matmul-optimization' already exists and is not an empty directory.
/content/cuda-matmul-optimization
a5a2b5f (HEAD -> main, origin/main, origin/HEAD) Added inference for full MNIST data set
4849cd1 Removed leftover referenceFile code
baaeca7 Implemented MLP model prediction in CUDA
00cecee Add MNIST CUDA inference and weights
6ea5e93 Added comparison to CPU matmult


In [ ]:
%cd /content/cuda-matmul-optimization
!git pull
!nvcc -arch=sm_75 mlp_benchmark.cu -o mlp_benchmark
!./mlp_benchmark


/content/cuda-matmul-optimization
